# Predictive Analytics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [1]:
from run_config import PATHS

In [2]:
import pandas as pd
import numpy as np
import datetime
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

## Preparations

In [3]:
INPUT = PATHS.train_test_dir

In [4]:
#GRID_SAMPLE = 35_000 # full or number
SPATIAL_UNIT = "COMMUNITY_AREAS" # HEXAGON
TIME_UNIT = "24H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [5]:
# "Settings" / Decisions for the training data

DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_TRAIN = INPUT / F"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"
    DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_{H3_RES}_VAL.parquet"

MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_demand",
    "date",
]

Load data and select features and target

In [6]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
val_df = pd.read_parquet(DATA_PATH_VAL)
test_df = pd.read_parquet(DATA_PATH_TEST)

In [7]:
val_df = val_df.sample(n=min(5_000, len(val_df)), random_state=42)

In [8]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)


y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'community_area', 'weather_station_distance_km', 'food_drink', 'landmark', 'shop', 'train_station', 'tmpc', 'relh', 'sknt', 'p01m', 'vsby', 'wind_dir_sin', 'wind_dir_cos', 'station_observed', 'weather_imputed', 'precipitation_missing', 'weather_rain', 'weather_snow', 'weather_fog_mist', 'weather_thunder', 'weather_freezing', 'precipitation_trace', 'weather_qc_corrected', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_SCT', 'skyc1_BKN', 'skyc1_OVC', 'skyc1_VV', 'weather_station_MDW', 'weather_station_ORD', 'weather_station_IGQ']
Target: uint32


In [9]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2026-04-13,4,1,0,1.0,6.123234e-17,0.000000,1.000000,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
1,2026-04-13,4,1,0,1.0,6.123234e-17,0.000000,1.000000,0.0,1.0,...,0.0,213.5,0.214142,0.0,39.0,10535.36,10.567061,3.75,117.00,Credit Card
2,2026-04-13,4,1,0,1.0,6.123234e-17,0.000000,1.000000,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
3,2026-04-13,4,1,0,1.0,6.123234e-17,0.000000,1.000000,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
4,2026-04-13,4,1,0,1.0,6.123234e-17,0.000000,1.000000,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
765,2026-04-23,4,4,0,1.0,6.123234e-17,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
766,2026-04-23,4,4,0,1.0,6.123234e-17,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
767,2026-04-22,4,3,0,1.0,6.123234e-17,0.974928,-0.222521,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
768,2026-04-23,4,4,0,1.0,6.123234e-17,0.433884,-0.900969,0.0,1.0,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips


In [10]:
model = SVR()

In [11]:
X_val = X_val

In [12]:
param_grid_linear = {
    "C": [1, 10],
    "epsilon": [0.1, 0.5, 1],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [1, 10],
    "epsilon": [0.1, 0.5],
    "kernel": ["rbf", "sigmoid"],
    "gamma": [0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["poly"],
    "degree": [3, 4, 5],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVR(),
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val, y_val)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.6173672510825833 best params: {'C': 10, 'epsilon': 0.5, 'kernel': 'linear'}
rbf_sigmoid best score: -0.040033584832487756 best params: {'C': 10, 'epsilon': 0.5, 'gamma': 0.01, 'kernel': 'sigmoid'}
poly best score: 0.46275389684029095 best params: {'C': 1, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.1, 'kernel': 'poly'}
Overall best: linear {'C': 10, 'epsilon': 0.5, 'kernel': 'linear'}


In [13]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 10, 'epsilon': 0.5, 'kernel': 'linear'}
Best CV score: 0.6173672510825833


In [14]:
# Train SVR with the best hyperparameters found by grid search
best_params = grid_search.best_params_
model = SVR(**best_params)
model.fit(X_train, y_train)

,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,10
,epsilon,0.5
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [15]:
# Make prediction 
y_pred = model.predict(X_test)

In [16]:
y_pred

array([-1.24309299e+00, -3.23637655e-03, -3.49001427e-03, -1.24309586e+00,
       -3.13038778e-03,  3.61514913e-01, -1.24341923e+00, -1.26197032e+00,
       -3.23924559e-03, -1.24309951e+00, -1.24301617e+00, -3.15955979e-03,
       -3.24289642e-03, -3.56262333e-03, -1.24307884e+00, -1.24304300e+00,
       -1.24334663e+00, -3.18638665e-03, -1.24298700e+00, -3.22223088e-03,
       -1.26206834e+00,  1.41248038e+03, -1.26178069e+00,  3.61425432e-01,
       -1.26205980e+00, -1.26218803e+00, -1.26207580e+00,  8.10475329e+00,
       -1.26229080e+00,  3.61194432e-01,  3.61416891e-01,  3.61466312e-01,
        3.61409436e-01,  3.61704546e-01, -1.26195182e+00, -1.26201892e+00,
        6.48126805e+00,  3.61533411e-01,  3.61297202e-01,  1.41410387e+03,
        1.73775666e+03,  7.56631852e-01,  3.61493676e-01, -1.26205016e+00,
        1.73899652e+03,  3.61387767e-01,  3.61528235e-01, -1.26196978e+00,
       -1.26199156e+00, -1.26209747e+00,  3.61435073e-01,  3.61515450e-01,
        1.99648846e+00, -

In [17]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 33.95519041815856
MSE: 26158.12665442473
RMSE: 161.73474164329917
R2 Score: 0.8712503342947199
